In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pickle

from math import log, sqrt
from time import time
from pprint import pprint

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score as AUC, log_loss, accuracy_score as accuracy
from sklearn.metrics import (mean_squared_error as MSE, mean_absolute_error as MAE, r2_score as R2,
                             explained_variance_score as EVS)
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler, MaxAbsScaler

from keras.models import Sequential
from keras.layers.core import Dense, Dropout
from keras.layers.normalization import BatchNormalization as BatchNorm
from keras.callbacks import EarlyStopping, ModelCheckpoint
from keras.layers.advanced_activations import *
from keras.models import load_model

%load_ext autoreload
%autoreload 2

%matplotlib inline

plt.rcParams['figure.figsize'] = (10, 8)

/home/bulent/anaconda3/lib/python3.6/site-packages/h5py/__init__.py:34: FutureWarning: Conversion of the second argument of issubdtype from `float` to `np.floating` is deprecated. In future, it will be treated as `np.float64 == np.dtype(float).type`.
  from ._conv import register_converters as _register_converters
Using TensorFlow backend.


In [2]:
with open('stations-6to31.pkl', 'rb') as f:
    datas = pickle.load(f)
    
x_train6_ = datas['x_train6']
y_train6 = datas['y_train6']
x_train_ = datas['x_train']
y_train = datas['y_train']
x_dev_ = datas['x_dev']
y_dev = datas['y_dev']
x_test_ = datas['x_test']
y_test = datas['y_test']

print(f'x_train shape: {x_train_.shape}, y_train shape: {y_train.shape}')
print(f'x_train6 shape: {x_train6_.shape}, y_train6 shape: {y_train6.shape}')
print(f'x_dev shape: {x_dev_.shape}, y_dev shape: {y_dev.shape}')
print(f'x_test shape: {x_test_.shape}, y_test shape: {y_test.shape}')

x_train shape: (52416, 5), y_train shape: (52416,)
x_train6 shape: (10654, 5), y_train6 shape: (10654,)
x_dev shape: (9011, 5), y_dev shape: (9011,)
x_test shape: (16899, 5), y_test shape: (16899,)


From the best 21 configurations modes of respective categories are as follows.

**Initializer:** normal

**Layers:** 2

**Batch Size:** 64

**Optimizer:** adamax

**Shuffle:** True

**Scaler:** RobustScaler

**Loss:** mean_absolute_error

In [3]:
def scale_data(scaler, datas):
    # scaler is a scaling function from sklearn library
    # datas is a dictionary, containing 3 sets of x_data with keys - x_train, x_dev, x_test
    # fit on x_train and return the transformed sets of data
    
    x_train = scaler.fit_transform(datas['x_train'].astype(float))
    x_dev = scaler.transform(datas['x_dev'].astype(float))
    x_test = scaler.transform(datas['x_test'].astype(float))
    
    transformed = {'x_train': x_train, 'x_dev': x_dev, 'x_test': x_test}
    return transformed

data6_ = {'x_train': x_train6_, 'x_dev': x_dev_, 'x_test': x_test_}
data_ = {'x_train': x_train_, 'x_dev': x_dev_, 'x_test': x_test_}

data6 = scale_data(RobustScaler(), data6_)
data = scale_data(RobustScaler(), data_)

x_train6 = data6['x_train']
x_train = data['x_train']

x_dev6 = data6['x_dev']
x_dev = data['x_dev']

x_test6 = data6['x_test']
x_test = data['x_test']

print(f'x_train shape: {x_train.shape}, y_train shape: {y_train.shape}')
print(f'x_train6 shape: {x_train6.shape}, y_train6 shape: {y_train6.shape}')
print(f'x_dev shape: {x_dev.shape}, y_dev shape: {y_dev.shape}')
print(f'x_test shape: {x_test.shape}, y_test shape: {y_test.shape}')

x_train shape: (52416, 5), y_train shape: (52416,)
x_train6 shape: (10654, 5), y_train6 shape: (10654,)
x_dev shape: (9011, 5), y_dev shape: (9011,)
x_test shape: (16899, 5), y_test shape: (16899,)


In [4]:
def radstimator(h1=20, h2=15, num_vars=5):
    init = 'normal'
    
    model = Sequential()
    model.add( Dense( h1, kernel_initializer=init, input_dim=num_vars ))
    model.add( PReLU( alpha_initializer=init ))
    model.add( BatchNorm())
    model.add( Dense( h2, kernel_initializer=init ))
    model.add( PReLU( alpha_initializer=init ))
    model.add( Dropout( rate=0.4 ))
    
    model.add( Dense( 1, kernel_initializer=init, activation='linear' ))
    
    return model

In [9]:
print(x_train6_[:5]) # 'Latitude', 'BSH', 'Temperature(avg)', 'Daylength', 'H0'

[[40.141       5.9         4.22916667  9.19499685 13.69287119]
 [40.141       1.3         7.6375      9.20626964 13.74607822]
 [40.141       0.          6.2375      9.21860396 13.80432176]
 [40.141       0.          3.32916667  9.23198817 13.86758193]
 [40.141       0.7         5.06956522  9.24640976 13.93583675]]


In [5]:
validation_data6 = ( x_dev6, y_dev )
for i in range(50):
    rads = radstimator(20, 15, 5)
    rads.compile(optimizer='adamax', loss='mean_absolute_error')

    early_stopping = EarlyStopping( monitor = 'val_loss', patience = 10, verbose = 0 )
    filepath = './6to31stats-4vars-h/6stations-h-nNTPhiH0 {}.h5'.format(i+1)
    checkpointer = ModelCheckpoint(filepath, monitor='val_loss', verbose=0, save_best_only=True )
    history = rads.fit( x_train6, y_train6, epochs = 250, batch_size = 64, shuffle = True, 
                         validation_data = validation_data6, callbacks = [ early_stopping, checkpointer ], verbose=0)

    p = rads.predict( x_train6, batch_size = 64 )

    mse = MSE( y_train6, p )
    rmse = sqrt( mse )
    mae = MAE( y_train6, p )
    r2 = R2( y_train6, p )
    evs = EVS( y_train6, p )
    
    print('C{:02d} » RMSE: {:.4f}, MAE: {:.4f}, R2: {:.4f}, EVS: {:.4f}, '.format(i+1,rmse, mae, r2, evs))

C01 » RMSE: 3.8732, MAE: 2.1992, R2: 0.8245, EVS: 0.8272, 
C02 » RMSE: 3.8400, MAE: 2.0876, R2: 0.8275, EVS: 0.8283, 
C03 » RMSE: 3.8624, MAE: 2.2026, R2: 0.8255, EVS: 0.8276, 
C04 » RMSE: 3.8788, MAE: 2.2284, R2: 0.8240, EVS: 0.8265, 
C05 » RMSE: 3.8496, MAE: 2.0648, R2: 0.8267, EVS: 0.8271, 
C06 » RMSE: 3.8583, MAE: 2.2389, R2: 0.8259, EVS: 0.8295, 
C07 » RMSE: 3.8993, MAE: 2.2590, R2: 0.8222, EVS: 0.8264, 
C08 » RMSE: 3.8150, MAE: 2.1275, R2: 0.8298, EVS: 0.8317, 
C09 » RMSE: 3.7976, MAE: 2.0294, R2: 0.8313, EVS: 0.8324, 
C10 » RMSE: 3.8913, MAE: 2.2849, R2: 0.8229, EVS: 0.8273, 
C11 » RMSE: 3.8420, MAE: 2.1522, R2: 0.8274, EVS: 0.8294, 
C12 » RMSE: 3.8474, MAE: 2.0469, R2: 0.8269, EVS: 0.8271, 
C13 » RMSE: 3.8435, MAE: 2.0602, R2: 0.8272, EVS: 0.8281, 
C14 » RMSE: 3.8180, MAE: 2.1388, R2: 0.8295, EVS: 0.8312, 
C15 » RMSE: 3.8514, MAE: 2.1335, R2: 0.8265, EVS: 0.8287, 
C16 » RMSE: 3.8557, MAE: 2.1412, R2: 0.8261, EVS: 0.8284, 
C17 » RMSE: 3.8305, MAE: 2.2077, R2: 0.8284, EVS: 0.8320

In [6]:
validation_data = ( x_dev, y_dev )
for i in range(50):
    rads = radstimator(20, 15, 5)
    rads.compile(optimizer='adamax', loss='mean_absolute_error')

    early_stopping = EarlyStopping( monitor = 'val_loss', patience = 10, verbose = 0 )
    filepath = './6to31stats-4vars-h/31stations-h-nNTPhiH0 {}.h5'.format(i+1)
    checkpointer = ModelCheckpoint(filepath, monitor='val_loss', verbose=0, save_best_only=True )
    history = rads.fit( x_train, y_train, epochs = 250, batch_size = 64, shuffle = True, 
                         validation_data = validation_data, callbacks = [ early_stopping, checkpointer ], verbose=0)

    p = rads.predict( x_train, batch_size = 64 )

    mse = MSE( y_train, p )
    rmse = sqrt( mse )
    mae = MAE( y_train, p )
    r2 = R2( y_train, p )
    evs = EVS( y_train, p )
    
    print('C{:02d} » RMSE: {:.4f}, MAE: {:.4f}, R2: {:.4f}, EVS: {:.4f}, '.format(i+1,rmse, mae, r2, evs))

C01 » RMSE: 3.0167, MAE: 1.7420, R2: 0.8778, EVS: 0.8806, 
C02 » RMSE: 2.9946, MAE: 1.6532, R2: 0.8796, EVS: 0.8798, 
C03 » RMSE: 3.0708, MAE: 1.8724, R2: 0.8734, EVS: 0.8786, 
C04 » RMSE: 2.9921, MAE: 1.6183, R2: 0.8798, EVS: 0.8798, 
C05 » RMSE: 2.9925, MAE: 1.7009, R2: 0.8797, EVS: 0.8805, 
C06 » RMSE: 2.9965, MAE: 1.6832, R2: 0.8794, EVS: 0.8797, 
C07 » RMSE: 2.9843, MAE: 1.6850, R2: 0.8804, EVS: 0.8808, 
C08 » RMSE: 3.0302, MAE: 1.7249, R2: 0.8767, EVS: 0.8771, 
C09 » RMSE: 2.9899, MAE: 1.6866, R2: 0.8800, EVS: 0.8804, 
C10 » RMSE: 3.0025, MAE: 1.6566, R2: 0.8789, EVS: 0.8790, 
C11 » RMSE: 3.0229, MAE: 1.6538, R2: 0.8773, EVS: 0.8779, 
C12 » RMSE: 2.9734, MAE: 1.6352, R2: 0.8813, EVS: 0.8814, 
C13 » RMSE: 3.0189, MAE: 1.7400, R2: 0.8776, EVS: 0.8795, 
C14 » RMSE: 2.9930, MAE: 1.6736, R2: 0.8797, EVS: 0.8798, 
C15 » RMSE: 2.9961, MAE: 1.6805, R2: 0.8795, EVS: 0.8796, 
C16 » RMSE: 3.0311, MAE: 1.7494, R2: 0.8766, EVS: 0.8771, 
C17 » RMSE: 3.0255, MAE: 1.7516, R2: 0.8771, EVS: 0.8783